In [0]:

CREATE OR REPLACE TABLE furlenco_analytics.materialized_tables.delivered_revenue_base_query as 

WITH base_return_item AS (
  SELECT
    items.id                                              AS item_id,
    items.order_id,
    items.user_details:displayId::STRING                  AS fur_id,           -- [1]
    items.user_id,
    items.display_id                                      AS item_display_id,
    ord.display_id                                        AS ord_display_id,
    items.activation_date,
    items.created_at + INTERVAL 330 MINUTES               AS item_created_at,  -- [4]
    MIN(CASE WHEN return_items.state <> 'CANCELLED'
            THEN return_items.created_at + INTERVAL 330 MINUTES END)  AS return_date,
    MIN(CASE WHEN rtp_orders.state NOT IN ('CANCELLED')
            THEN rtp_orders.created_at + INTERVAL 330 MINUTES END)    AS rent_to_purchase_date
  FROM furlenco_silver.order_management_systems_evolve.items AS items
  LEFT JOIN furlenco_silver.order_management_systems_evolve.orders AS ord
    ON items.order_id = ord.id
  LEFT JOIN furlenco_silver.order_management_systems_evolve.return_items AS return_items
    ON items.id = return_items.item_id
   AND return_items.state <> 'CANCELLED'
  LEFT JOIN furlenco_silver.order_management_systems_evolve.rent_to_purchase_items AS rtp_items
    ON items.id = rtp_items.item_id
  LEFT JOIN furlenco_silver.order_management_systems_evolve.rent_to_purchase_orders AS rtp_orders
    ON rtp_items.rent_to_purchase_order_id = rtp_orders.id
   AND rtp_orders.state NOT IN ('CANCELLED')
  WHERE items.vertical = 'FURLENCO_RENTAL'
    AND items.state NOT IN ('CANCELLED')
    AND ord.state NOT IN ('CANCELLED', 'AWAITING_PAYMENT')
  GROUP BY
    items.id,
    items.order_id,
    items.user_details:displayId::STRING,           -- [1]
    items.user_id,
    items.activation_date,
    (items.created_at + INTERVAL 330 MINUTES),      -- [4]
    items.display_id,
    ord.display_id
),

new_acquisitions AS (
  SELECT
    user_id, item_id, order_id,
    fur_id, activation_date, item_created_at,
    COALESCE(return_date, rent_to_purchase_date)              AS termination_date,
    DENSE_RANK() OVER (PARTITION BY user_id ORDER BY order_id) AS rnk,
    item_display_id, ord_display_id
  FROM base_return_item
),

has_previous_item AS (
  SELECT
    *,
    MAX(termination_date) OVER (
      PARTITION BY user_id
      ORDER BY order_id, termination_date DESC
      ROWS BETWEEN UNBOUNDED PRECEDING AND 1 PRECEDING
    ) AS previous_termination_date,
    CASE WHEN EXISTS (
      SELECT 1
      FROM base_return_item b
      WHERE b.user_id = new_acquisitions.user_id
        AND b.order_id < new_acquisitions.order_id
        AND COALESCE(b.return_date, b.rent_to_purchase_date) IS NULL
    ) THEN 1 ELSE 0 END AS has_active_previous_items_corr
  FROM new_acquisitions
),

-- [5] Split into two CTEs: window function computed first, then alias is safe to reference
acquisition_types_base AS (
  SELECT
    *,
    MIN(previous_termination_date) OVER (PARTITION BY user_id, order_id) AS max_previous_termination_date
  FROM has_previous_item
),

acquisition_types AS (
  SELECT
    fur_id, user_id, order_id, item_id, activation_date,
    item_created_at, termination_date,
    has_active_previous_items_corr,
    max_previous_termination_date,
    CASE WHEN (
      (item_created_at > max_previous_termination_date
        AND max_previous_termination_date IS NOT NULL
        AND has_active_previous_items_corr = 0)
      OR rnk = 1
    ) THEN 'New' ELSE 'Upsell' END AS acquisition_type,
    rnk, item_display_id, ord_display_id
  FROM acquisition_types_base
),

-- Identify active subscription groups (New orders and their subsequent Upsells)
subscription_groups AS (
  SELECT
    a.*,
    SUM(CASE WHEN a.acquisition_type = 'New' THEN 1 ELSE 0 END) OVER (
      PARTITION BY a.user_id
      ORDER BY a.order_id
      ROWS UNBOUNDED PRECEDING
    ) AS acquisition_group_id
  FROM acquisition_types a
),

-- Calculating ASB for each subscription group
asb_calculation AS (
  SELECT
    s.*,
    FIRST_VALUE(order_id) OVER (
      PARTITION BY user_id, acquisition_group_id
    ) AS first_order_in_group
  FROM subscription_groups s
),

acquisitions AS (
  SELECT
    fur_id, order_id, item_id, activation_date,
    item_created_at, termination_date,
    acquisition_type,
    acquisition_group_id,
    first_order_in_group,
    MIN(activation_date) OVER (PARTITION BY fur_id, first_order_in_group) AS acquisition_date,
    item_display_id, ord_display_id
  FROM asb_calculation
),

----------------------------------------------------------------------------------------------
-- Delivered Revenue Query
----------------------------------------------------------------------------------------------

base_item AS (
  SELECT
    fur_id,
    order_id,
    item_id  AS accountable_entity_id,
    'ITEM'   AS accountable_entity_type,
    activation_date, first_order_in_group, acquisition_date, acquisition_type
  FROM acquisitions

  UNION ALL

  SELECT DISTINCT
    at.user_details:displayId::STRING  AS fur_id,   -- [1]
    at.order_id,
    at.id                              AS accountable_entity_id,
    'ATTACHMENT'                       AS accountable_entity_type,
    at.activation_date,
    acq.first_order_in_group,
    acq.acquisition_date,
    acq.acquisition_type
  FROM acquisitions AS acq
  LEFT JOIN furlenco_silver.order_management_systems_evolve.attachments AS at
    ON acq.order_id = at.order_id
  WHERE at.id IS NOT NULL
),

--------------------------------------------------------------------------------------------------
-- Item and attachment VAS
--------------------------------------------------------------------------------------------------

vas_base_item_attachment AS (
  SELECT
    vas.id,
    vas.entity_id,
    vas.entity_type,
    vas.user_action_type,
    vas.state,
    vas.name,
    vas.type,
    i.user_details:displayId::STRING   AS fur_id,  -- [1]
    vas.start_date,
    vas.end_date,
    ROUND(DATEDIFF(day, start_date, end_date) / 30.45) AS tenures,
    vas.created_at,
    i.order_id
  FROM furlenco_silver.order_management_systems_evolve.value_added_services AS vas
  JOIN furlenco_silver.order_management_systems_evolve.items AS i
    ON i.id = vas.entity_id
  WHERE vas.entity_type = 'ITEM'
    AND vas.state NOT IN ('CANCELLED')

  UNION ALL

  SELECT
    vas.id,
    vas.entity_id,
    vas.entity_type,
    vas.user_action_type,
    vas.state,
    vas.name,
    vas.type,
    a.user_details:displayId::STRING   AS fur_id,  -- [1]
    vas.start_date,
    vas.end_date,
    ROUND(DATEDIFF(day, start_date, end_date) / 30.45) AS tenures,
    vas.created_at,
    a.order_id
  FROM furlenco_silver.order_management_systems_evolve.value_added_services AS vas
  JOIN furlenco_silver.order_management_systems_evolve.attachments AS a
    ON a.id = vas.entity_id
  WHERE vas.entity_type = 'ATTACHMENT'
    AND vas.state NOT IN ('CANCELLED')
),

vas_info AS (
  SELECT
    v.id                                                      AS vas_id,
    v.state                                                   AS vas_state,
    v.order_id,
    v.entity_id,
    v.entity_type,
    v.start_date                                              AS activation_date,
    v.fur_id,
    v.start_date,
    v.end_date,
    v.created_at + INTERVAL 330 MINUTES                       AS vas_created_at,  -- [4]
    rr.monetary_components:tax.total::DOUBLE
      / NULLIF(v.tenures::DOUBLE, 0)                          AS total_tax,        -- [1][2]
    rr.monetary_components:taxableAmount::DOUBLE              AS taxable_amount,   -- [1][2]
    CASE
      WHEN v.type IN ('DELIVERY_CHARGE', 'AC_INSTALLATION_CHARGE')
        THEN rr.monetary_components:taxableAmount::DOUBLE
      ELSE rr.monetary_components:taxableAmount::DOUBLE
           / NULLIF(v.tenures::DOUBLE, 0)
    END                                                       AS splitted_,        -- [1][2]
    rr.monetary_components:taxableAmount::DOUBLE
      / NULLIF(v.tenures::DOUBLE, 0)                          AS splitted_taxableamount, -- [1][2]
    rr.vertical,
    rr.monetary_components:discounts[0].code::STRING          AS code_1,           -- [1]
    rr.monetary_components:discounts[0].amount::DOUBLE        AS code_1_amount,    -- [1][2]
    rr.monetary_components:discounts[1].code::STRING          AS code_2,           -- [1]
    rr.monetary_components:discounts[1].amount::DOUBLE        AS code_2_amount     -- [1][2]
  FROM vas_base_item_attachment AS v
  JOIN furlenco_silver.furbooks_evolve.revenue_recognitions AS rr
    ON rr.accountable_entity_id = v.id
   AND rr.accountable_entity_type = 'VALUE_ADDED_SERVICE'
   AND v.user_action_type = 'CART_CHECKOUT'
  WHERE rr.vertical = 'FURLENCO_RENTAL'         -- [6] was unqualified; vertical lives on rr
    AND rr.state NOT IN ('CANCELLED', 'INVALIDATED')
    AND rr.deleted_at IS NULL                    -- [7]
),

final_output_delivered_revenue AS (
  SELECT DISTINCT
    b.*,
    rr.start_date,
    rr.monetary_components:taxableAmount::DOUBLE                        AS taxable_amount,    -- [1][2]
    rr.monetary_components:tax.total::DOUBLE                            AS total_tax,         -- [1][2]
    (rr.monetary_components:taxableAmount::DOUBLE
     + rr.monetary_components:tax.total::DOUBLE)                        AS taxable_tax_amount,
    rr.monetary_components:discounts[0].code::STRING                    AS code_1,            -- [1]
    rr.monetary_components:discounts[0].amount::DOUBLE                  AS code_1_amount,     -- [1][2]
    rr.monetary_components:discounts[1].code::STRING                    AS code_2,            -- [1]
    rr.monetary_components:discounts[1].amount::DOUBLE                  AS code_2_amount      -- [1][2]
  FROM base_item AS b
  JOIN furlenco_silver.furbooks_evolve.revenue_recognitions AS rr
    ON b.accountable_entity_id = rr.accountable_entity_id
   AND b.accountable_entity_type = rr.accountable_entity_type
  WHERE rr.state NOT IN ('CANCELLED', 'INVALIDATED')
    AND rr.deleted_at IS NULL                    -- [7]
  QUALIFY DENSE_RANK() OVER (
    PARTITION BY b.fur_id, rr.accountable_entity_id, rr.accountable_entity_type
    ORDER BY rr.start_date
  ) = 1

  UNION ALL

  SELECT DISTINCT
    v.fur_id,
    v.order_id,
    v.vas_id,
    'VALUE_ADDED_SERVICE'                      AS accountable_entity_type,
    v.activation_date,
    acq.first_order_in_group,
    acq.acquisition_date,
    acq.acquisition_type,
    v.start_date,
    v.splitted_taxableamount                   AS taxable_amount,
    v.total_tax,
    (v.splitted_taxableamount + v.total_tax)   AS taxable_tax_amount,
    v.code_1,
    v.code_1_amount,
    v.code_2,
    v.code_2_amount
  FROM vas_info AS v
  JOIN acquisitions AS acq
    ON v.order_id = acq.order_id
)

SELECT DISTINCT
  * EXCEPT (first_order_in_group),              -- [3] Redshift EXCLUDE → Databricks EXCEPT
  CURRENT_TIMESTAMP() AS refreshed_at
FROM final_output_delivered_revenue
WHERE 1=1
